### <span style=color:blue>PA3 SOLUTION for PART 1 </span>

In [1]:
import sys
import json
import csv
import yaml

import importlib

import math

import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

from datetime import time
from datetime import date
from datetime import datetime
# with the above choices, the imported datetime.time(2023,07,01) is recognized
# from datetime import date
# from datetime import datetime

import pprint

import psycopg2
from sqlalchemy import create_engine, text as sql_text

# Create an utilities file util.py in a folder benchmarking and import it
# NOTE: I moved my util.py to the directory "helper_functions" -- seems like a better name
sys.path.append('ECS116-HELPER-FUNCTIONS')
import util

import importlib

<span style=color:blue>Getting mongodb connection set up</span>

In [2]:
from pymongo import MongoClient

client = MongoClient()
# could have written client = MongoClient("localhost", 27017)
#                 or client = MongoClient("mongodb://localhost:27017/")

<span style=color:blue>Setting up collection "listings" in mongodb</span>

In [3]:
# I have (or will have) a database "airbnb"
db = client.airbnb

# inside the "airbnb" database, I have (or will have) a collection "listings"
# Note: the collection listings is not created until I insert an element into it
listings = db.listings
print(db.list_collection_names())

# I may have some other collections in my airbnb database...

['listings_small', 'listings_test', 'listings', 'listings_with_reviews_duplicate', 'calendar']


<span style=color:blue>Query 1: Output is the number of listings that have last review between February 1, 2021, and March
15, 2023, inclusive </span>

In [5]:
dt1 = datetime.strptime('2021-02-01','%Y-%m-%d')
dt2 = datetime.strptime('2023-03-15','%Y-%m-%d')

count = db.listings.count_documents({'$and': [{'last_review' : {'$gte': dt1}}, 
                                              {'last_review' : {'$lte': dt2}}]})

print(count)

2467


<span style=color:blue>Query 2: Output is the number of listings that have an array of reviews with length at least 50 </span>

In [6]:
# imitating https://stackoverflow.com/questions/41918605/mongodb-find-array-length-greater-than-specified-size

count = db.listings.count_documents({
    'reviews.49': { 
        '$exists': True 
    }
})
print(count)

5488


<span style=color:blue>Query 3: Output is the number of listings that have a review containing the word ”awesome” (case
sensitive) OR a review containing the word ”amazing” (case sensitive).    </span>

In [7]:
count = listings.count_documents({
    '$or' : [{
        'reviews.comments' : {
            '$regex':  '^.*awesome.*$'  
        }
    } , {
        'reviews.comments' : { 
            '$regex':  '^.*amazing.*$'    
        } 
    }
            ]
})

print(count)

14966


<span style=color:blue>Query 4 Output is the number of listings that have a review containing the word ”awesome” (case
insensitive) OR a review containing the word ”amazing” (case insensitive).    </span>

In [8]:
count = listings.count_documents({
    '$or' : [{
        'reviews.comments' : {
            '$regex':  '^.*awesome.*$', '$options': 'i'
        }
    } , {
        'reviews.comments' : { 
            '$regex':  '^.*amazing.*$', '$options': 'i'
        } 
    }
            ]
})

print(count)

16077


### <span style=color:blue> Scratch paper    </span>

In [26]:
out = db.listings.find({
    'reviews.comments': pattern
})
i = 0
for doc in out:
    if i < 5:
        pprint.pp(doc)
        print()
        i += 1
        
    

{'_id': ObjectId('682ba4868c0cc07552f33142'),
 'id': '1000637914058448005',
 'name': 'Heart of Manhattan walk to all',
 'host_id': '541397825',
 'host_name': 'Tusnoda',
 'neighbourhood_cleansed': 'Midtown',
 'neighbourhood_group_cleansed': 'Manhattan',
 'latitude': 40.74382,
 'longitude': -73.98259,
 'room_type': 'Entire home/apt',
 'price': '$387.00',
 'minimum_nights': 4,
 'number_of_reviews': 25,
 'last_review': datetime.datetime(2025, 2, 9, 0, 0),
 'reviews_per_month': 1.72,
 'calculated_host_listings_count': 1,
 'has_availability': 't',
 'number_of_reviews_ltm': 23,
 'license': 'Exempt',
 'reviews': [{'listing_id': '1000637914058448005',
              'review_id': '1053028410173715980',
              'date': datetime.datetime(2023, 12, 23, 0, 0),
              'reviewer_id': '435509160',
              'reviewer_name': 'Glenda',
              'comments': "The location was incredible!! It's in walking "
                          "distance to so many great places and there's a sub "


In [20]:
count = db.listings.count_documents({
    'reviews': {
        '$elemMatch': {
            'comments': {
                '$regex': r'\bawesome\b',
                '$options': 'i'  # case-insensitive
            }
        }
    }
})
print(count)


372


In [21]:
out = db.listings.find({
    'reviews': {
        '$elemMatch': {
            'comments': {
                '$regex': r'\bawesome\b',
                '$options': 'i'  # case-insensitive
            }
        }
    }
})
i = 0
for doc in out:
    if i < 5:
        pprint.pp(doc)
        print()
        i += 1
        

{'_id': ObjectId('682ba4868c0cc07552f33142'),
 'id': '1000637914058448005',
 'name': 'Heart of Manhattan walk to all',
 'host_id': '541397825',
 'host_name': 'Tusnoda',
 'neighbourhood_cleansed': 'Midtown',
 'neighbourhood_group_cleansed': 'Manhattan',
 'latitude': 40.74382,
 'longitude': -73.98259,
 'room_type': 'Entire home/apt',
 'price': '$387.00',
 'minimum_nights': 4,
 'number_of_reviews': 25,
 'last_review': datetime.datetime(2025, 2, 9, 0, 0),
 'reviews_per_month': 1.72,
 'calculated_host_listings_count': 1,
 'has_availability': 't',
 'number_of_reviews_ltm': 23,
 'license': 'Exempt',
 'reviews': [{'listing_id': '1000637914058448005',
              'review_id': '1053028410173715980',
              'date': datetime.datetime(2023, 12, 23, 0, 0),
              'reviewer_id': '435509160',
              'reviewer_name': 'Glenda',
              'comments': "The location was incredible!! It's in walking "
                          "distance to so many great places and there's a sub "
